# Experiment B: translate-pivot for MSA / Egyptian / Levantine (SemEval 2027 Task 6, Task 2)

The fine-tuned Qwen3.5-4B (S2 adapter) answers the English-translated questions in English (translation of questions and of
answers happens locally via OpenRouter; this notebook is only the GPU step). Input dataset: `mmcqa-exp-b-inputs`.
Download `exp_b_results.zip` from the Output tab. **Settings -> Accelerator -> GPU T4 x2, Internet on.**

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-experiments.git"
BRANCH = "main"
ROWS = 100
WORK = "/kaggle/working/results"

In [ ]:
import glob, os, subprocess, sys
CODE = "/kaggle/working/mmcqa-experiments"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi -L
!pip install -q -r {CODE}/requirements/qwen.txt
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|bitsandbytes|accelerate|huggingface-hub|tokenizers) "

In [ ]:
try:                                   # optional: a Kaggle secret HF_TOKEN avoids anonymous download limits
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: continuing with anonymous downloads")

In [ ]:
!cd {CODE} && python experiments/exp_b_translate_pivot/run_kaggle.py --workdir {WORK} --inputs /kaggle/input --rows {ROWS}

In [ ]:
import json, glob
print(open(f"{WORK}/logs/exp_b_completion.json").read()[:2500])
for p in sorted(glob.glob(f"{WORK}/predictions/pivot_en_s2/dev_*.jsonl")):
    print(p); [print("  ", json.loads(l)["question"], "->", json.loads(l)["answer"]) for l in open(p).readlines()[:3]]

In [ ]:
ZIPNAME = "exp_b_results.zip"

import shutil, pathlib
media = pathlib.Path(os.environ.get("MMCQA_MEDIA", "/kaggle/temp/mmcqa_media"))
for tmp in (WORK + "/mmcultureqa", "/kaggle/working/mmcqa_media", "/kaggle/working/mmcqa_envs", str(media)):
    shutil.rmtree(tmp, ignore_errors=True)
!cd /kaggle/working && zip -qr {ZIPNAME} results && ls -lh {ZIPNAME}